In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Lab 1: ML Metadata (MLMD) — Enhanced Walkthrough\n",
    "\n",
    "## Tracking a Real Sklearn Pipeline with Lineage Visualization\n",
    "\n",
    "In this lab, you will learn how to use **ML Metadata (MLMD)** to record and retrieve metadata\n",
    "associated with a real machine learning workflow. Unlike a basic walkthrough that only registers\n",
    "placeholder artifacts, this lab:\n",
    "\n",
    "1. **Trains a real scikit-learn model** (Random Forest on the Iris dataset)\n",
    "2. **Records every stage** — data ingestion, preprocessing, training, and evaluation — in an MLMD metadata store\n",
    "3. **Tracks lineage** between datasets, transformed data, trained models, and evaluation metrics\n",
    "4. **Visualizes the full lineage DAG** so you can trace any artifact back to its origins\n",
    "\n",
    "### Prerequisites\n",
    "```bash\n",
    "pip install ml-metadata scikit-learn matplotlib networkx\n",
    "```\n",
    "\n",
    "### MLMD Data Model Refresher\n",
    "\n",
    "| Concept | Description |\n",
    "|---|---|\n",
    "| **ArtifactType** | Schema for a category of artifacts (e.g., Dataset, Model) |\n",
    "| **Artifact** | A specific instance — a file, a model, a metric result |\n",
    "| **ExecutionType** | Schema for a pipeline step (e.g., Trainer, Evaluator) |\n",
    "| **Execution** | A specific run of that step |\n",
    "| **Event** | Links an Artifact to an Execution as INPUT or OUTPUT |\n",
    "| **Context** | Groups artifacts & executions (e.g., an Experiment) |"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 1 — Setup & Imports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Install dependencies (uncomment if needed)\n",
    "# !pip install ml-metadata scikit-learn matplotlib networkx"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import json\n",
    "import datetime\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# MLMD imports\n",
    "from ml_metadata import metadata_store\n",
    "from ml_metadata.proto import metadata_store_pb2\n",
    "\n",
    "# Sklearn imports\n",
    "from sklearn.datasets import load_iris\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.metrics import accuracy_score, f1_score, classification_report\n",
    "import joblib\n",
    "\n",
    "# Visualization imports\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.patches as mpatches\n",
    "import networkx as nx\n",
    "\n",
    "print('Setup complete!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 2 — Initialize the Metadata Store\n",
    "\n",
    "We use a **fake (in-memory) database** for fast experimentation.\n",
    "In production, you would swap this for SQLite or MySQL."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "connection_config = metadata_store_pb2.ConnectionConfig()\n",
    "connection_config.fake_database.SetInParent()\n",
    "store = metadata_store.MetadataStore(connection_config)\n",
    "\n",
    "print('Metadata store initialized (in-memory).')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 3 — Register Artifact Types\n",
    "\n",
    "We define four artifact types for our pipeline:\n",
    "- **RawDataset** — the original data before any processing\n",
    "- **TransformedDataset** — data after preprocessing (scaling, splitting)\n",
    "- **Model** — a trained sklearn model saved to disk\n",
    "- **ModelMetrics** — evaluation metrics (accuracy, F1, etc.)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- RawDataset ---\n",
    "raw_data_type = metadata_store_pb2.ArtifactType()\n",
    "raw_data_type.name = 'RawDataset'\n",
    "raw_data_type.properties['name'] = metadata_store_pb2.STRING\n",
    "raw_data_type.properties['num_samples'] = metadata_store_pb2.INT\n",
    "raw_data_type.properties['num_features'] = metadata_store_pb2.INT\n",
    "raw_data_type_id = store.put_artifact_type(raw_data_type)\n",
    "\n",
    "# --- TransformedDataset ---\n",
    "transformed_data_type = metadata_store_pb2.ArtifactType()\n",
    "transformed_data_type.name = 'TransformedDataset'\n",
    "transformed_data_type.properties['name'] = metadata_store_pb2.STRING\n",
    "transformed_data_type.properties['split'] = metadata_store_pb2.STRING\n",
    "transformed_data_type.properties['num_samples'] = metadata_store_pb2.INT\n",
    "transformed_data_type_id = store.put_artifact_type(transformed_data_type)\n",
    "\n",
    "# --- Model ---\n",
    "model_type = metadata_store_pb2.ArtifactType()\n",
    "model_type.name = 'Model'\n",
    "model_type.properties['name'] = metadata_store_pb2.STRING\n",
    "model_type.properties['framework'] = metadata_store_pb2.STRING\n",
    "model_type.properties['version'] = metadata_store_pb2.INT\n",
    "model_type.properties['hyperparameters'] = metadata_store_pb2.STRING\n",
    "model_type_id = store.put_artifact_type(model_type)\n",
    "\n",
    "# --- ModelMetrics ---\n",
    "metrics_type = metadata_store_pb2.ArtifactType()\n",
    "metrics_type.name = 'ModelMetrics'\n",
    "metrics_type.properties['accuracy'] = metadata_store_pb2.DOUBLE\n",
    "metrics_type.properties['f1_weighted'] = metadata_store_pb2.DOUBLE\n",
    "metrics_type.properties['report'] = metadata_store_pb2.STRING\n",
    "metrics_type_id = store.put_artifact_type(metrics_type)\n",
    "\n",
    "print('Registered Artifact Types:')\n",
    "for at in store.get_artifact_types():\n",
    "    print(f'  - {at.name} (id={at.id})')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 4 — Register Execution Types\n",
    "\n",
    "We define three execution types matching our pipeline steps:\n",
    "1. **DataIngestion** — loading the raw dataset\n",
    "2. **Preprocessing** — splitting and scaling\n",
    "3. **Training** — fitting the model and evaluating"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def register_execution_type(store, name):\n",
    "    \"\"\"Helper to register an ExecutionType with a 'state' property.\"\"\"\n",
    "    exec_type = metadata_store_pb2.ExecutionType()\n",
    "    exec_type.name = name\n",
    "    exec_type.properties['state'] = metadata_store_pb2.STRING\n",
    "    type_id = store.put_execution_type(exec_type)\n",
    "    return type_id\n",
    "\n",
    "ingestion_type_id = register_execution_type(store, 'DataIngestion')\n",
    "preprocessing_type_id = register_execution_type(store, 'Preprocessing')\n",
    "training_type_id = register_execution_type(store, 'Training')\n",
    "\n",
    "print('Registered Execution Types:')\n",
    "for et in store.get_execution_types():\n",
    "    print(f'  - {et.name} (id={et.id})')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 5 — Helper Functions\n",
    "\n",
    "These utility functions reduce boilerplate when creating executions and events."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def create_execution(store, type_id, state='RUNNING'):\n",
    "    \"\"\"Create and register an execution, returning its ID.\"\"\"\n",
    "    execution = metadata_store_pb2.Execution()\n",
    "    execution.type_id = type_id\n",
    "    execution.properties['state'].string_value = state\n",
    "    [execution_id] = store.put_executions([execution])\n",
    "    return execution_id\n",
    "\n",
    "\n",
    "def complete_execution(store, execution_id, type_id):\n",
    "    \"\"\"Mark an execution as COMPLETED.\"\"\"\n",
    "    execution = metadata_store_pb2.Execution()\n",
    "    execution.id = execution_id\n",
    "    execution.type_id = type_id\n",
    "    execution.properties['state'].string_value = 'COMPLETED'\n",
    "    store.put_executions([execution])\n",
    "\n",
    "\n",
    "def register_event(store, artifact_id, execution_id, event_type):\n",
    "    \"\"\"Record an event (INPUT or OUTPUT) linking an artifact to an execution.\"\"\"\n",
    "    event = metadata_store_pb2.Event()\n",
    "    event.artifact_id = artifact_id\n",
    "    event.execution_id = execution_id\n",
    "    event.type = event_type\n",
    "    store.put_events([event])\n",
    "\n",
    "\n",
    "print('Helper functions defined.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 6 — Pipeline Step 1: Data Ingestion\n",
    "\n",
    "We load the Iris dataset and register it as a **RawDataset** artifact.\n",
    "An **DataIngestion** execution is created and linked via events."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ---- Actual ML work ----\n",
    "iris = load_iris()\n",
    "X, y = iris.data, iris.target\n",
    "\n",
    "print(f'Loaded Iris dataset: {X.shape[0]} samples, {X.shape[1]} features')\n",
    "\n",
    "# ---- MLMD: Create execution ----\n",
    "ingestion_exec_id = create_execution(store, ingestion_type_id)\n",
    "\n",
    "# ---- MLMD: Register the raw dataset artifact ----\n",
    "raw_data_artifact = metadata_store_pb2.Artifact()\n",
    "raw_data_artifact.uri = 'in-memory://sklearn.datasets.load_iris'\n",
    "raw_data_artifact.type_id = raw_data_type_id\n",
    "raw_data_artifact.properties['name'].string_value = 'Iris Dataset'\n",
    "raw_data_artifact.properties['num_samples'].int_value = X.shape[0]\n",
    "raw_data_artifact.properties['num_features'].int_value = X.shape[1]\n",
    "[raw_data_id] = store.put_artifacts([raw_data_artifact])\n",
    "\n",
    "# ---- MLMD: Register output event ----\n",
    "register_event(store, raw_data_id, ingestion_exec_id,\n",
    "               metadata_store_pb2.Event.DECLARED_OUTPUT)\n",
    "\n",
    "# ---- MLMD: Complete execution ----\n",
    "complete_execution(store, ingestion_exec_id, ingestion_type_id)\n",
    "\n",
    "print(f'Data Ingestion complete. Raw dataset artifact ID: {raw_data_id}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 7 — Pipeline Step 2: Preprocessing\n",
    "\n",
    "We split the data into train/test sets and apply `StandardScaler`.\n",
    "This step takes the **RawDataset** as input and produces two\n",
    "**TransformedDataset** artifacts (train split + test split)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ---- Actual ML work ----\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42, stratify=y\n",
    ")\n",
    "\n",
    "scaler = StandardScaler()\n",
    "X_train_scaled = scaler.fit_transform(X_train)\n",
    "X_test_scaled = scaler.transform(X_test)\n",
    "\n",
    "print(f'Train set: {X_train_scaled.shape[0]} samples')\n",
    "print(f'Test set:  {X_test_scaled.shape[0]} samples')\n",
    "\n",
    "# ---- MLMD: Create execution ----\n",
    "preprocess_exec_id = create_execution(store, preprocessing_type_id)\n",
    "\n",
    "# ---- MLMD: Input event (raw data -> preprocessing) ----\n",
    "register_event(store, raw_data_id, preprocess_exec_id,\n",
    "               metadata_store_pb2.Event.DECLARED_INPUT)\n",
    "\n",
    "# ---- MLMD: Register transformed train artifact ----\n",
    "train_artifact = metadata_store_pb2.Artifact()\n",
    "train_artifact.uri = 'in-memory://iris_train_scaled'\n",
    "train_artifact.type_id = transformed_data_type_id\n",
    "train_artifact.properties['name'].string_value = 'Iris Train (scaled)'\n",
    "train_artifact.properties['split'].string_value = 'train'\n",
    "train_artifact.properties['num_samples'].int_value = X_train_scaled.shape[0]\n",
    "[train_artifact_id] = store.put_artifacts([train_artifact])\n",
    "\n",
    "# ---- MLMD: Register transformed test artifact ----\n",
    "test_artifact = metadata_store_pb2.Artifact()\n",
    "test_artifact.uri = 'in-memory://iris_test_scaled'\n",
    "test_artifact.type_id = transformed_data_type_id\n",
    "test_artifact.properties['name'].string_value = 'Iris Test (scaled)'\n",
    "test_artifact.properties['split'].string_value = 'test'\n",
    "test_artifact.properties['num_samples'].int_value = X_test_scaled.shape[0]\n",
    "[test_artifact_id] = store.put_artifacts([test_artifact])\n",
    "\n",
    "# ---- MLMD: Output events ----\n",
    "register_event(store, train_artifact_id, preprocess_exec_id,\n",
    "               metadata_store_pb2.Event.DECLARED_OUTPUT)\n",
    "register_event(store, test_artifact_id, preprocess_exec_id,\n",
    "               metadata_store_pb2.Event.DECLARED_OUTPUT)\n",
    "\n",
    "# ---- MLMD: Complete execution ----\n",
    "complete_execution(store, preprocess_exec_id, preprocessing_type_id)\n",
    "\n",
    "print(f'Preprocessing complete. Train artifact ID: {train_artifact_id}, Test artifact ID: {test_artifact_id}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 8 — Pipeline Step 3: Training & Evaluation\n",
    "\n",
    "We train a **RandomForestClassifier**, evaluate it on the test set,\n",
    "and register both the **Model** and **ModelMetrics** artifacts.\n",
    "\n",
    "Notice how hyperparameters are stored as a JSON string property on the Model artifact."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ---- Actual ML work ----\n",
    "hyperparams = {'n_estimators': 100, 'max_depth': 5, 'random_state': 42}\n",
    "clf = RandomForestClassifier(**hyperparams)\n",
    "clf.fit(X_train_scaled, y_train)\n",
    "\n",
    "y_pred = clf.predict(X_test_scaled)\n",
    "acc = accuracy_score(y_test, y_pred)\n",
    "f1 = f1_score(y_test, y_pred, average='weighted')\n",
    "report = classification_report(y_test, y_pred, target_names=iris.target_names)\n",
    "\n",
    "print(f'Accuracy: {acc:.4f}')\n",
    "print(f'F1 (weighted): {f1:.4f}')\n",
    "print(f'\\nClassification Report:\\n{report}')\n",
    "\n",
    "# Save model to disk\n",
    "os.makedirs('artifacts', exist_ok=True)\n",
    "model_path = 'artifacts/iris_rf_model.joblib'\n",
    "joblib.dump(clf, model_path)\n",
    "print(f'Model saved to {model_path}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ---- MLMD: Create training execution ----\n",
    "training_exec_id = create_execution(store, training_type_id)\n",
    "\n",
    "# ---- MLMD: Input events (train + test data -> training) ----\n",
    "register_event(store, train_artifact_id, training_exec_id,\n",
    "               metadata_store_pb2.Event.DECLARED_INPUT)\n",
    "register_event(store, test_artifact_id, training_exec_id,\n",
    "               metadata_store_pb2.Event.DECLARED_INPUT)\n",
    "\n",
    "# ---- MLMD: Register model artifact ----\n",
    "model_artifact = metadata_store_pb2.Artifact()\n",
    "model_artifact.uri = model_path\n",
    "model_artifact.type_id = model_type_id\n",
    "model_artifact.properties['name'].string_value = 'Iris RandomForest v1'\n",
    "model_artifact.properties['framework'].string_value = 'scikit-learn'\n",
    "model_artifact.properties['version'].int_value = 1\n",
    "model_artifact.properties['hyperparameters'].string_value = json.dumps(hyperparams)\n",
    "[model_artifact_id] = store.put_artifacts([model_artifact])\n",
    "\n",
    "# ---- MLMD: Register metrics artifact ----\n",
    "metrics_artifact = metadata_store_pb2.Artifact()\n",
    "metrics_artifact.uri = 'in-memory://eval_metrics_v1'\n",
    "metrics_artifact.type_id = metrics_type_id\n",
    "metrics_artifact.properties['accuracy'].double_value = acc\n",
    "metrics_artifact.properties['f1_weighted'].double_value = f1\n",
    "metrics_artifact.properties['report'].string_value = report\n",
    "[metrics_artifact_id] = store.put_artifacts([metrics_artifact])\n",
    "\n",
    "# ---- MLMD: Output events ----\n",
    "register_event(store, model_artifact_id, training_exec_id,\n",
    "               metadata_store_pb2.Event.DECLARED_OUTPUT)\n",
    "register_event(store, metrics_artifact_id, training_exec_id,\n",
    "               metadata_store_pb2.Event.DECLARED_OUTPUT)\n",
    "\n",
    "# ---- MLMD: Complete execution ----\n",
    "complete_execution(store, training_exec_id, training_type_id)\n",
    "\n",
    "print(f'Training complete.')\n",
    "print(f'  Model artifact ID:   {model_artifact_id}')\n",
    "print(f'  Metrics artifact ID: {metrics_artifact_id}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 9 — Group Everything Under an Experiment Context\n",
    "\n",
    "We create an **Experiment** context and associate all artifacts and executions with it.\n",
    "This lets us query \"everything that happened in this experiment\" later."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ---- Register ContextType ----\n",
    "experiment_ctx_type = metadata_store_pb2.ContextType()\n",
    "experiment_ctx_type.name = 'Experiment'\n",
    "experiment_ctx_type.properties['note'] = metadata_store_pb2.STRING\n",
    "experiment_ctx_type.properties['timestamp'] = metadata_store_pb2.STRING\n",
    "experiment_ctx_type_id = store.put_context_type(experiment_ctx_type)\n",
    "\n",
    "# ---- Create Context instance ----\n",
    "experiment_ctx = metadata_store_pb2.Context()\n",
    "experiment_ctx.type_id = experiment_ctx_type_id\n",
    "experiment_ctx.name = 'iris-classification-exp-1'\n",
    "experiment_ctx.properties['note'].string_value = (\n",
    "    'Iris classification with RandomForest, StandardScaler preprocessing'\n",
    ")\n",
    "experiment_ctx.properties['timestamp'].string_value = (\n",
    "    datetime.datetime.now().isoformat()\n",
    ")\n",
    "[experiment_ctx_id] = store.put_contexts([experiment_ctx])\n",
    "\n",
    "# ---- Attributions (artifacts -> context) ----\n",
    "attributions = []\n",
    "for aid in [raw_data_id, train_artifact_id, test_artifact_id,\n",
    "            model_artifact_id, metrics_artifact_id]:\n",
    "    attr = metadata_store_pb2.Attribution()\n",
    "    attr.artifact_id = aid\n",
    "    attr.context_id = experiment_ctx_id\n",
    "    attributions.append(attr)\n",
    "\n",
    "# ---- Associations (executions -> context) ----\n",
    "associations = []\n",
    "for eid in [ingestion_exec_id, preprocess_exec_id, training_exec_id]:\n",
    "    assoc = metadata_store_pb2.Association()\n",
    "    assoc.execution_id = eid\n",
    "    assoc.context_id = experiment_ctx_id\n",
    "    associations.append(assoc)\n",
    "\n",
    "store.put_attributions_and_associations(attributions, associations)\n",
    "\n",
    "print(f'Experiment context created: \"{experiment_ctx.name}\" (id={experiment_ctx_id})')\n",
    "print(f'  Linked {len(attributions)} artifacts and {len(associations)} executions.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 10 — Querying the Metadata Store\n",
    "\n",
    "Now that we have recorded everything, let's query the store to answer\n",
    "real-world questions you might ask in production."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Question 1: What artifacts exist in this experiment?\n",
    "print('='*60)\n",
    "print('Artifacts in experiment:')\n",
    "print('='*60)\n",
    "for a in store.get_artifacts_by_context(experiment_ctx_id):\n",
    "    a_type = store.get_artifact_types_by_id([a.type_id])[0]\n",
    "    print(f'  [{a_type.name}] id={a.id}  uri={a.uri}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Question 2: Which dataset was used to train the model?\n",
    "# Strategy: Start from the Model artifact, trace back through events.\n",
    "print('='*60)\n",
    "print('Tracing lineage: Model -> Training inputs -> Raw data')\n",
    "print('='*60)\n",
    "\n",
    "# Find the model\n",
    "model = store.get_artifacts_by_type('Model')[0]\n",
    "print(f'\\n1. Found model: \"{model.properties[\"name\"].string_value}\" (id={model.id})')\n",
    "\n",
    "# Find the execution that produced this model\n",
    "model_events = store.get_events_by_artifact_ids([model.id])\n",
    "output_event = [e for e in model_events if e.type == metadata_store_pb2.Event.DECLARED_OUTPUT][0]\n",
    "training_exec = store.get_executions_by_id([output_event.execution_id])[0]\n",
    "exec_type_name = store.get_execution_types_by_id([training_exec.type_id])[0].name\n",
    "print(f'2. Produced by execution: \"{exec_type_name}\" (id={training_exec.id})')\n",
    "\n",
    "# Find all inputs to that execution\n",
    "exec_events = store.get_events_by_execution_ids([training_exec.id])\n",
    "input_ids = [e.artifact_id for e in exec_events if e.type == metadata_store_pb2.Event.DECLARED_INPUT]\n",
    "input_artifacts = store.get_artifacts_by_id(input_ids)\n",
    "print(f'3. Training inputs:')\n",
    "for a in input_artifacts:\n",
    "    a_type = store.get_artifact_types_by_id([a.type_id])[0]\n",
    "    print(f'   - [{a_type.name}] \"{a.properties[\"name\"].string_value}\" (id={a.id})')\n",
    "\n",
    "# Trace further back: where did the train data come from?\n",
    "train_data_events = store.get_events_by_artifact_ids([input_artifacts[0].id])\n",
    "output_ev = [e for e in train_data_events if e.type == metadata_store_pb2.Event.DECLARED_OUTPUT][0]\n",
    "preproc_exec = store.get_executions_by_id([output_ev.execution_id])[0]\n",
    "preproc_events = store.get_events_by_execution_ids([preproc_exec.id])\n",
    "raw_ids = [e.artifact_id for e in preproc_events if e.type == metadata_store_pb2.Event.DECLARED_INPUT]\n",
    "raw_artifacts = store.get_artifacts_by_id(raw_ids)\n",
    "print(f'4. Original raw data:')\n",
    "for a in raw_artifacts:\n",
    "    a_type = store.get_artifact_types_by_id([a.type_id])[0]\n",
    "    print(f'   - [{a_type.name}] \"{a.properties[\"name\"].string_value}\" '\n",
    "          f'({a.properties[\"num_samples\"].int_value} samples)')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Question 3: What were the hyperparameters and metrics for the model?\n",
    "print('='*60)\n",
    "print('Model details')\n",
    "print('='*60)\n",
    "model = store.get_artifacts_by_type('Model')[0]\n",
    "print(f'Name:            {model.properties[\"name\"].string_value}')\n",
    "print(f'Framework:       {model.properties[\"framework\"].string_value}')\n",
    "print(f'Hyperparameters: {model.properties[\"hyperparameters\"].string_value}')\n",
    "\n",
    "metrics = store.get_artifacts_by_type('ModelMetrics')[0]\n",
    "print(f'\\nAccuracy:        {metrics.properties[\"accuracy\"].double_value:.4f}')\n",
    "print(f'F1 (weighted):   {metrics.properties[\"f1_weighted\"].double_value:.4f}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 11 — Lineage Visualization\n",
    "\n",
    "This is the key addition over the base lab. We build a **directed acyclic graph (DAG)**\n",
    "from all events in the metadata store and render it with `networkx` + `matplotlib`.\n",
    "\n",
    "The graph shows:\n",
    "- **Blue rectangles** = Artifacts\n",
    "- **Orange rounded boxes** = Executions\n",
    "- **Arrows** = data flow (INPUT edges go into executions, OUTPUT edges come out)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def build_lineage_graph(store):\n",
    "    \"\"\"\n",
    "    Build a NetworkX DiGraph from all artifacts, executions, and events\n",
    "    in the metadata store.\n",
    "    \"\"\"\n",
    "    G = nx.DiGraph()\n",
    "    \n",
    "    # Add artifact nodes\n",
    "    for a in store.get_artifacts():\n",
    "        a_type = store.get_artifact_types_by_id([a.type_id])[0]\n",
    "        name = a.properties.get('name')\n",
    "        label = name.string_value if name and name.string_value else a.uri\n",
    "        split = a.properties.get('split')\n",
    "        if split and split.string_value:\n",
    "            label += f'\\n({split.string_value})'\n",
    "        node_id = f'a_{a.id}'\n",
    "        G.add_node(node_id, label=f'{a_type.name}\\n{label}',\n",
    "                   node_type='artifact', artifact_type=a_type.name)\n",
    "    \n",
    "    # Add execution nodes\n",
    "    for e in store.get_executions():\n",
    "        e_type = store.get_execution_types_by_id([e.type_id])[0]\n",
    "        node_id = f'e_{e.id}'\n",
    "        state = e.properties['state'].string_value if 'state' in e.properties else ''\n",
    "        G.add_node(node_id, label=f'{e_type.name}\\n[{state}]',\n",
    "                   node_type='execution')\n",
    "    \n",
    "    # Add edges from events\n",
    "    all_artifacts = store.get_artifacts()\n",
    "    all_artifact_ids = [a.id for a in all_artifacts]\n",
    "    all_events = store.get_events_by_artifact_ids(all_artifact_ids)\n",
    "    \n",
    "    for ev in all_events:\n",
    "        a_node = f'a_{ev.artifact_id}'\n",
    "        e_node = f'e_{ev.execution_id}'\n",
    "        if ev.type in (metadata_store_pb2.Event.INPUT,\n",
    "                       metadata_store_pb2.Event.DECLARED_INPUT):\n",
    "            G.add_edge(a_node, e_node)\n",
    "        elif ev.type in (metadata_store_pb2.Event.OUTPUT,\n",
    "                         metadata_store_pb2.Event.DECLARED_OUTPUT):\n",
    "            G.add_edge(e_node, a_node)\n",
    "    \n",
    "    return G\n",
    "\n",
    "\n",
    "def visualize_lineage(G, figsize=(16, 10)):\n",
    "    \"\"\"\n",
    "    Render the lineage DAG with distinct styles for artifacts and executions.\n",
    "    \"\"\"\n",
    "    # Use a layered layout for DAG\n",
    "    for layer, nodes in enumerate(nx.topological_generations(G)):\n",
    "        for node in nodes:\n",
    "            G.nodes[node]['subset'] = layer\n",
    "    pos = nx.multipartite_layout(G, subset_key='subset', align='horizontal')\n",
    "    \n",
    "    # Separate node types\n",
    "    artifact_nodes = [n for n, d in G.nodes(data=True) if d['node_type'] == 'artifact']\n",
    "    exec_nodes = [n for n, d in G.nodes(data=True) if d['node_type'] == 'execution']\n",
    "    \n",
    "    labels = nx.get_node_attributes(G, 'label')\n",
    "    \n",
    "    fig, ax = plt.subplots(1, 1, figsize=figsize)\n",
    "    \n",
    "    # Draw artifact nodes (blue squares)\n",
    "    nx.draw_networkx_nodes(G, pos, nodelist=artifact_nodes,\n",
    "                           node_color='#4A90D9', node_shape='s',\n",
    "                           node_size=3500, alpha=0.9, ax=ax)\n",
    "    \n",
    "    # Draw execution nodes (orange circles)\n",
    "    nx.draw_networkx_nodes(G, pos, nodelist=exec_nodes,\n",
    "                           node_color='#F5A623', node_shape='o',\n",
    "                           node_size=3500, alpha=0.9, ax=ax)\n",
    "    \n",
    "    # Draw edges\n",
    "    nx.draw_networkx_edges(G, pos, edge_color='#555555',\n",
    "                           arrows=True, arrowsize=20,\n",
    "                           connectionstyle='arc3,rad=0.1', ax=ax)\n",
    "    \n",
    "    # Draw labels\n",
    "    nx.draw_networkx_labels(G, pos, labels, font_size=7,\n",
    "                            font_color='white', font_weight='bold', ax=ax)\n",
    "    \n",
    "    # Legend\n",
    "    legend_elements = [\n",
    "        mpatches.Patch(facecolor='#4A90D9', label='Artifact'),\n",
    "        mpatches.Patch(facecolor='#F5A623', label='Execution'),\n",
    "    ]\n",
    "    ax.legend(handles=legend_elements, loc='upper left', fontsize=11)\n",
    "    ax.set_title('ML Pipeline Lineage DAG (from MLMD)', fontsize=14, fontweight='bold')\n",
    "    plt.tight_layout()\n",
    "    plt.savefig('artifacts/lineage_dag.png', dpi=150, bbox_inches='tight')\n",
    "    plt.show()\n",
    "    print('Lineage DAG saved to artifacts/lineage_dag.png')\n",
    "\n",
    "\n",
    "print('Visualization functions defined.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Build and render the lineage graph\n",
    "G = build_lineage_graph(store)\n",
    "visualize_lineage(G)\n",
    "\n",
    "print(f'\\nGraph summary: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 12 — Lineage Trace: From Model Back to Raw Data\n",
    "\n",
    "This function recursively walks **backward** through events to find\n",
    "every upstream artifact that contributed to a given artifact.\n",
    "This is the core capability that makes MLMD valuable in production:\n",
    "if a model goes wrong, you can trace exactly which data and steps produced it."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def trace_lineage_back(store, artifact_id, depth=0, visited=None):\n",
    "    \"\"\"\n",
    "    Recursively trace an artifact's lineage back to its origins.\n",
    "    Returns a list of (depth, artifact_type_name, artifact_name, artifact_id) tuples.\n",
    "    \"\"\"\n",
    "    if visited is None:\n",
    "        visited = set()\n",
    "    if artifact_id in visited:\n",
    "        return []\n",
    "    visited.add(artifact_id)\n",
    "    \n",
    "    results = []\n",
    "    artifact = store.get_artifacts_by_id([artifact_id])[0]\n",
    "    a_type = store.get_artifact_types_by_id([artifact.type_id])[0]\n",
    "    name = artifact.properties.get('name')\n",
    "    a_name = name.string_value if name and name.string_value else artifact.uri\n",
    "    results.append((depth, a_type.name, a_name, artifact.id))\n",
    "    \n",
    "    # Find executions that produced this artifact\n",
    "    events = store.get_events_by_artifact_ids([artifact_id])\n",
    "    for ev in events:\n",
    "        if ev.type in (metadata_store_pb2.Event.OUTPUT,\n",
    "                       metadata_store_pb2.Event.DECLARED_OUTPUT):\n",
    "            # Find inputs to that execution\n",
    "            exec_events = store.get_events_by_execution_ids([ev.execution_id])\n",
    "            for exec_ev in exec_events:\n",
    "                if exec_ev.type in (metadata_store_pb2.Event.INPUT,\n",
    "                                    metadata_store_pb2.Event.DECLARED_INPUT):\n",
    "                    results.extend(\n",
    "                        trace_lineage_back(store, exec_ev.artifact_id, depth+1, visited)\n",
    "                    )\n",
    "    return results\n",
    "\n",
    "\n",
    "# Trace the model's full lineage\n",
    "print('Full lineage of the trained model:')\n",
    "print('='*60)\n",
    "model = store.get_artifacts_by_type('Model')[0]\n",
    "lineage = trace_lineage_back(store, model.id)\n",
    "\n",
    "for depth, type_name, name, aid in lineage:\n",
    "    indent = '  ' * depth\n",
    "    connector = 'ROOT' if depth == 0 else '<--'\n",
    "    print(f'{indent}{connector} [{type_name}] \"{name}\" (id={aid})')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Part 13 — Summary\n",
    "\n",
    "In this enhanced lab, you:\n",
    "\n",
    "1. **Initialized** an MLMD metadata store\n",
    "2. **Registered** artifact types (RawDataset, TransformedDataset, Model, ModelMetrics) and execution types (DataIngestion, Preprocessing, Training)\n",
    "3. **Ran a real sklearn pipeline** — loaded Iris data, scaled it, trained a RandomForest, and evaluated it\n",
    "4. **Recorded every step** as MLMD executions with proper input/output events\n",
    "5. **Grouped everything** under an Experiment context\n",
    "6. **Queried the store** to answer questions like \"which data trained this model?\"\n",
    "7. **Visualized the full lineage DAG** using networkx + matplotlib\n",
    "8. **Traced lineage recursively** from the model back to the raw dataset\n",
    "\n",
    "### Key Takeaways\n",
    "\n",
    "- MLMD acts as a **structured log** for your ML pipeline — every artifact and step is tracked\n",
    "- **Events** are the backbone of lineage — they connect artifacts to the executions that consumed or produced them\n",
    "- **Contexts** let you group related work (e.g., one experiment) for easy querying\n",
    "- In production, you would use **SQLite or MySQL** instead of the fake in-memory database\n",
    "- The lineage DAG gives you a visual map to **debug issues** and **audit model provenance**\n",
    "\n",
    "### Exercises for Students\n",
    "\n",
    "1. **Add a second experiment**: Train a different model (e.g., `SVC` or `LogisticRegression`), register it with new artifact/execution entries, and compare the two experiments by querying the metadata store.\n",
    "2. **Persist with SQLite**: Change the connection config to use `connection_config.sqlite.filename_uri` so the metadata survives across runs.\n",
    "3. **Add a data validation step**: Use TFDV (as shown in the original lab) to generate statistics and a schema, and register those as additional artifacts in the pipeline.\n",
    "4. **Extend the visualization**: Color-code artifact nodes by type (e.g., blue for data, green for models, red for metrics)."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}